# 11.4 圖片描述怎麼擴展成問答？


同一張紅圓形圖片，問「什麼顏色」應答紅，問「什麼形狀」應答圓。只訓練「紅色圓形」描述，目標每次固定，無法直接告訴我們模型是否會讀問題挑選屬性。本節把同圖多問放進資料，確保目標依賴圖片與問題兩者。

前置是[11.3圖片描述目標](https://birdhackor.github.io/tiny-perceptron-vlm/11.3.html)與[7.1對話資料](https://birdhackor.github.io/tiny-perceptron-vlm/7.1.html)。圖片問答英文叫Visual Question Answering，縮寫VQA。它不是只給圖片一個類別，而是回答該問題所要求的內容。資料必須保留實際圖片或可恢復圖片的生成條件，不能只喂問題與答案後聲稱模型用圖。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
records = [
    {"image": "red_circle", "question": "什麼顏色？", "answer": "紅"},
    {"image": "red_circle", "question": "什麼形狀？", "answer": "圓形"},
]
for row in records:
    print(row["image"], row["question"], "→", row["answer"])
print("同圖", records[0]["image"] == records[1]["image"])
print("答案不同", records[0]["answer"] != records[1]["answer"])

輸入兩筆字典，`image`這裡只是可讀的圖片身份名稱，不是像素。前三個字段分別記錄圖、題目、目標回答。輸出第一行紅圓圖顏色→紅，第二行同圖形狀→圓形，後面兩個比較均True。它讓作者看見「保持圖不變，只換問題」確實對應不同目標。

真正資料可以用[10.1的scene工具](https://birdhackor.github.io/tiny-perceptron-vlm/10.1.html)按red、circle生成該圖，並記錄大小與位移；這個工具按指定條件直接畫圖，沒有隨機選擇。更複雜的產圖工具若會隨機挑位置或背景，還要記seed（種子）：它是一個指定亂數選擇起點的整數，像抽籤機的起始號碼。例如產圖前設定42，就保存42；重建時用相同生成程式、參數與呼叫順序，從42開始重播這批選擇。生成配方與軟體版本也應一起保存，方便重建相同圖。將圖片標記與像素一起送入模型後，答案位置計算代價的方式仍如上一節，不需要為每種問句另造一個全新輸出系統。

[正式問答實驗](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/vqa.json)每張合成圖各問`shape?`與`color?`，要求只答`circle`、`square`或顏色名稱。六種顏色形狀各有三個訓練位置，共18張圖、36道問答；位移1的六張圖形成12道驗證題，位移2的六張圖形成12道最後測試題。同圖的兩問放在同一側，沒有拿訓練過的圖換問法當作新圖。這次留的是新位置，六種屬性組合仍已見過，不是未見組合的測試。

還要有同問題不同圖片。例如兩張都問顏色，一張紅圓一張藍圓，應回答紅與藍。只保留紅圓多問，模型可能不看圖片而記住「顏色題答紅、形狀題答圓」。把不同屬性組合配齊，並將新組合留作測試，才能同時檢查問題選擇與視覺使用；[11.8](https://birdhackor.github.io/tiny-perceptron-vlm/11.8.html)會具體討論替圖驗證。

例如若模型對同圖兩問都回「紅色圓形」，描述內容沒錯，卻未必遵循只要顏色或形狀的任務。評估應按問題要求判斷，不能因為答案含正確詞就全部算對。可以先用短而明確的目標紅、圓形，再用另一個同時詢問兩項的題允許完整描述。這樣資料和檢查都告訴模型「問題會改變要輸出的範圍」。同圖身份也讓作者追查是否錯誤配了另一張圖；保持記錄中的圖、問題與目標一並移動，別分別打亂三個清單。

練習增加第三筆：圖保持red_circle，問題改為「請同時給顏色與形狀」，目標寫「紅色圓形」。先預測哪些字段相同、哪些改變，再打印核對。若只換問題不換答案，你是在提供矛盾訓練；問題的要求應在目標內容裡留下可觀察的變化。
